# 03 — Quantization and Benchmarks

Convert the fine-tuned FP16 model to a **Q4_K_M GGUF** file, then benchmark:
- Disk size (FP16 vs Q4)
- Inference speed (CPU tokens/sec)
- Accuracy (on the test set)
- GPT-4o-mini baseline (for comparison)
- Raspberry Pi feasibility (theoretical)

## Prerequisites
- Notebook 2 completed (`models/merged-fp16/` exists)

## Runtime
- CPU or T4 fine
- ~15-30 min total (most of it is the Q4 quantization step)


## 0. Bootstrap

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = (
    'COLAB_GPU' in os.environ
    or 'COLAB_RELEASE_TAG' in os.environ
    or (Path('/content').exists() and not Path('/content').is_symlink())
)

if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/icd10-slm')
else:
    PROJECT = Path.cwd()
    while not (PROJECT / 'requirements.txt').exists():
        if PROJECT.parent == PROJECT:
            raise RuntimeError(f"Couldn't find repo root from {Path.cwd()}")
        PROJECT = PROJECT.parent

assert PROJECT.exists(), f"Project not found at {PROJECT}"
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

# Load secrets — .env from /content (Path B) or from project
if not os.getenv('HF_TOKEN'):
    try:
        from dotenv import load_dotenv
    except ImportError:
        !pip install -q python-dotenv
        from dotenv import load_dotenv
    import shutil
    candidates = [
        Path('/content/.env'),
        Path('/content/drive/MyDrive/icd10-slm/.env'),
    ]
    for c in candidates:
        if c.exists():
            if c != Path('/content/.env'):
                shutil.copy(c, '/content/.env')
            load_dotenv('/content/.env')
            break

assert os.getenv('HF_TOKEN'), "HF_TOKEN missing"
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

from src import config
print(f"✓ env:     {'Colab' if IN_COLAB else 'Local'}")
print(f"✓ project: {PROJECT}")
print(f"✓ model:   {config.BASE_MODEL}")


In [ ]:
# Verify Notebook 2 ran
assert (config.MERGED_DIR / 'config.json').exists(), \
    "merged-fp16 not found — run Notebook 2 first"

fp16_size_gb = sum(
    f.stat().st_size for f in config.MERGED_DIR.rglob('*') if f.is_file()
) / 1024 / 1024 / 1024
print(f"✓ FP16 model: {fp16_size_gb:.2f} GB at {config.MERGED_DIR}")


## 1. Install llama.cpp (for GGUF conversion + CPU inference)

We clone llama.cpp and build it. The conversion tool is pure Python; the
inference runtime is compiled C++. First install is ~3-5 min.


In [ ]:
import os
from pathlib import Path

LLAMA_CPP = Path('/content/llama.cpp') if IN_COLAB else PROJECT / '.llama.cpp'

if not LLAMA_CPP.exists():
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git {LLAMA_CPP}

# Install the Python conversion requirements
!pip install -q -r {LLAMA_CPP}/requirements.txt

# Build the quantize binary
if not (LLAMA_CPP / 'build' / 'bin' / 'llama-quantize').exists():
    !cd {LLAMA_CPP} && cmake -B build -DGGML_CUDA=OFF > /dev/null 2>&1
    !cd {LLAMA_CPP} && cmake --build build --config Release -j -t llama-quantize llama-cli > /dev/null 2>&1

print(f"✓ llama.cpp ready at {LLAMA_CPP}")


## 2. Convert FP16 → GGUF-FP16 (intermediate)

llama.cpp's quantizer needs a GGUF file as input. We first convert our
merged-fp16 safetensors to GGUF-FP16, then quantize that to Q4_K_M.


In [ ]:
gguf_fp16 = config.MODELS_DIR / 'merged-fp16.gguf'

if not gguf_fp16.exists():
    !python {LLAMA_CPP}/convert_hf_to_gguf.py \
        {config.MERGED_DIR} \
        --outfile {gguf_fp16} \
        --outtype f16

fp16_gguf_mb = gguf_fp16.stat().st_size / 1024 / 1024
print(f"✓ GGUF FP16: {fp16_gguf_mb:.1f} MB  →  {gguf_fp16}")


## 3. Quantize FP16-GGUF → Q4_K_M

Q4_K_M is the most popular 4-bit quantization. 4-bit weights with some
6-bit outliers in the attention layers. Best quality-for-size tradeoff.


In [ ]:
if not config.GGUF_PATH.exists():
    !{LLAMA_CPP}/build/bin/llama-quantize {gguf_fp16} {config.GGUF_PATH} Q4_K_M

q4_mb = config.GGUF_PATH.stat().st_size / 1024 / 1024
compression = fp16_gguf_mb / q4_mb
print(f"✓ Q4_K_M:     {q4_mb:.1f} MB  →  {config.GGUF_PATH}")
print(f"  Compression: {compression:.2f}×")

# Can delete the intermediate FP16 GGUF to save space
if gguf_fp16.exists():
    gguf_fp16.unlink()
    print(f"  (deleted intermediate {gguf_fp16.name})")


## 4. Benchmark — disk size

In [ ]:
from src.benchmarks import measure_disk_size, pi_feasibility_check

sizes = {
    'FP16 (HF format)':  measure_disk_size(config.MERGED_DIR),
    'Q4_K_M (GGUF)':     measure_disk_size(config.GGUF_PATH),
}

print("💾 DISK SIZE")
print("=" * 50)
for name, mb in sizes.items():
    print(f"  {name:22s}  {mb:>7,.1f} MB   ({mb/1024:.2f} GB)")
print(f"  Compression ratio:    {sizes['FP16 (HF format)']/sizes['Q4_K_M (GGUF)']:.2f}×")


## 5. Load Q4 model via llama-cpp-python for inference

In [ ]:
!pip install -q llama-cpp-python

from llama_cpp import Llama

llm = Llama(
    model_path=str(config.GGUF_PATH),
    n_ctx=512,
    n_threads=4,
    verbose=False,
)

# Test a prediction
messages = [
    {"role": "system", "content": config.SYSTEM_PROMPT},
    {"role": "user",   "content": "Type 2 diabetes mellitus without complications"},
]
r = llm.create_chat_completion(messages=messages, max_tokens=16, temperature=0)
print(f"Q4 test prediction: {r['choices'][0]['message']['content'].strip()!r}")


## 6. Benchmark — CPU inference speed

In [ ]:
import time
import json
from src.benchmarks import benchmark_generation

# Load a handful of test prompts
with open(config.TEST_PATH) as f:
    test_rows = [json.loads(line) for line in f]

prompts = [ex['clinical_text'] for ex in test_rows[:20]]

def gen_fn(prompt, max_tokens):
    messages = [
        {"role": "system", "content": config.SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    r = llm.create_chat_completion(messages=messages, max_tokens=max_tokens, temperature=0)
    out = r['choices'][0]['message']['content']
    n_tok = r['usage']['completion_tokens']
    return out, n_tok

q4_bench = benchmark_generation(gen_fn, prompts, max_new_tokens=16, warmup=2)

print("⚡ Q4 CPU INFERENCE SPEED")
print("=" * 50)
print(f"  Tokens/sec:              {q4_bench['tokens_per_second']:>7.2f}")
print(f"  First-token latency:     {q4_bench['first_token_latency_ms']:>7.1f} ms")
print(f"  Total tokens generated:  {q4_bench['total_tokens']:>7}")
print(f"  Total time:              {q4_bench['total_time_s']:>7.2f} s")


## 7. Full accuracy eval — Q4 vs GPT-4o-mini

We run the Q4 model and gpt-4o-mini on a sample of 500 test examples
(full 5,084 would take ~30 min at CPU speeds — 500 is a representative slice).


In [ ]:
import random
from src.eval import evaluate
import pandas as pd

# Load codebook for is_valid_code check
codebook = pd.read_parquet(config.CODEBOOK_PATH)
valid_codes = set(codebook['code'].str.upper())

# Sample 500 test examples (seeded for reproducibility)
random.seed(42)
test_sample = random.sample(test_rows, min(500, len(test_rows)))
test_inputs = [{'input': ex['clinical_text'], 'gold': ex['code']} for ex in test_sample]

# --- Q4 model ----------------------------------------------------------
def predict_q4(text):
    messages = [
        {"role": "system", "content": config.SYSTEM_PROMPT},
        {"role": "user",   "content": text},
    ]
    r = llm.create_chat_completion(messages=messages, max_tokens=16, temperature=0)
    return r['choices'][0]['message']['content'].strip()

print("Evaluating Q4 model on 500 test examples...")
res_q4 = evaluate(predict_q4, test_inputs, valid_codes, 'FT-Q4')
print(f"  exact match:         {res_q4.exact_match*100:.1f}%")
print(f"  category (3-char):   {res_q4.category_match*100:.1f}%")
print(f"  valid format:        {res_q4.valid_format*100:.1f}%")
print(f"  valid in codebook:   {res_q4.valid_in_codebook*100:.1f}%")
print(f"  avg latency:         {res_q4.avg_latency_ms:.0f} ms")


## 8. GPT-4o-mini baseline (our yardstick)

In [ ]:
from openai import OpenAI
client = OpenAI()

def predict_gpt4omini(text):
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {"role": "system", "content": config.SYSTEM_PROMPT},
            {"role": "user",   "content": text},
        ],
        max_tokens=16,
        temperature=0,
    )
    return r.choices[0].message.content.strip()

print("Evaluating GPT-4o-mini on 500 test examples (costs ~$0.05)...")
res_gpt = evaluate(predict_gpt4omini, test_inputs, valid_codes, 'GPT-4o-mini')
print(f"  exact match:         {res_gpt.exact_match*100:.1f}%")
print(f"  category (3-char):   {res_gpt.category_match*100:.1f}%")
print(f"  valid format:        {res_gpt.valid_format*100:.1f}%")
print(f"  valid in codebook:   {res_gpt.valid_in_codebook*100:.1f}%")
print(f"  avg latency:         {res_gpt.avg_latency_ms:.0f} ms")


## 9. Summary table — the money shot for the presentation

In [ ]:
# Build a comparison dataframe
import pandas as pd

rows = []
for r in [res_q4, res_gpt]:
    rows.append({
        'Model':               r.config,
        'Exact match':         f"{r.exact_match*100:.1f}%",
        'Category match':      f"{r.category_match*100:.1f}%",
        'Valid format':        f"{r.valid_format*100:.1f}%",
        'Valid code':          f"{r.valid_in_codebook*100:.1f}%",
        'Latency (ms)':        f"{r.avg_latency_ms:.0f}",
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

# Save results for later
(config.MODELS_DIR / 'benchmark_results.json').write_text(
    pd.DataFrame(rows).to_json(orient='records', indent=2)
)
print(f"\n✓ results saved to models/benchmark_results.json")


## 10. Raspberry Pi feasibility check

In [ ]:
from src.benchmarks import pi_feasibility_check

q4_size_mb = config.GGUF_PATH.stat().st_size / 1024 / 1024

print("🍓 RASPBERRY PI FEASIBILITY")
print("=" * 50)
for pi_ram in [2, 4, 8]:
    r = pi_feasibility_check(q4_size_mb, pi_ram_gb=pi_ram)
    verdict = '✓ fits' if r['fits'] else '✗ too big'
    print(f"  Pi with {pi_ram} GB RAM:  {verdict}  (headroom: {r['headroom_mb']:.0f} MB)")

# Extrapolate tokens/sec
print(f"\nCPU inference speed measured here:  {q4_bench['tokens_per_second']:.1f} tok/s")
print(f"  Extrapolated Pi 4 (~40-60% speed): {q4_bench['tokens_per_second']*0.5:.1f} tok/s")
print(f"  Extrapolated Pi 5 (~70-90% speed): {q4_bench['tokens_per_second']*0.8:.1f} tok/s")
print(f"\nFor a typical ~8-token ICD-10 code output:")
print(f"  Pi 4 expected latency:  {8 / (q4_bench['tokens_per_second']*0.5):.1f} sec")
print(f"  Pi 5 expected latency:  {8 / (q4_bench['tokens_per_second']*0.8):.1f} sec")


## ✅ Done

You now have:
- `models/merged-q4.gguf` — quantized model (~1.1 GB)
- `models/benchmark_results.json` — performance numbers for the presentation

### Key findings (fill in after running)

| Finding | Expected |
|---|---|
| Q4 is ~3× smaller than FP16 | ✅ |
| Q4 accuracy close to FP16 | ✅ (within 2-5%) |
| Q4 beats GPT-4o-mini on this narrow domain | usually ✅ |
| Q4 fits on Pi 4+ | ✅ |

### Next
Open `04_agent_chatbot.ipynb` — we wire up RAG, tools, and the Streamlit demo.
